# Phase 4 — Cross-Generator AI Image Detection Experiments

This notebook executes the full suite of Phase 4 experiments:
1. **Experiment 1 (Binary Baseline):** Train and evaluate models (Logistic Regression, RBF SVM, Random Forest, LightGBM) on Real vs. All Fake generators.
2. **Experiment 2 (Within-Generator / Diagonal):** Train and test matching generator pairs (Real + Gen_X).
3. **Experiment 3 (Cross-Generator Transfer Matrix):** 5×5 train-generator vs test-generator transfer evaluation.

In [ ]:
# Cell 1: Mount Google Drive & Setup Path
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    repo_path = Path('/content/drive/MyDrive/ml_project/cross-generator-ai-image-detection')
    if not repo_path.exists():
        repo_path = Path('/content/cross-generator-ai-image-detection')
    if repo_path.exists():
        sys.path.insert(0, str(repo_path / 'src'))
except ImportError:
    sys.path.insert(0, str(Path.cwd().parent / 'src'))
    sys.path.insert(0, str(Path.cwd() / 'src'))

print("Python path updated.")

In [ ]:
# Cell 2: Dependencies Installation
!pip install -q lightgbm scikit-learn seaborn matplotlib pandas numpy

In [ ]:
# Cell 3: Data Sanity Check
import numpy as np
from experiments.data_loader import load_subset, FEATURES_ROOT

print(f"Features Root: {FEATURES_ROOT}")
try:
    X_tr, y_tr, g_tr = load_subset("train")
    print(f"✓ Train loaded: X={X_tr.shape}, y={y_tr.shape}, Generators={np.unique(g_tr)}")
    X_filtered, y_filtered, g_filtered = load_subset("train", generators=["Real", "SDXL"])
    print(f"✓ Filtered (Real + SDXL) loaded: X={X_filtered.shape}, y={y_filtered.shape}, Generators={np.unique(g_filtered)}")
except Exception as e:
    print(f"Sanity check notice: {e} (Ensure features exist at FEATURES_ROOT prior to running loader)")

In [ ]:
# Cell 4: Run Experiment 1 — Binary Baseline (Real vs All Fake)
from experiments.exp1_binary import run_exp1

exp1_results = run_exp1()
display(exp1_results["results_df"])

In [ ]:
# Cell 5: Run Experiment 2 — Within-Generator (Diagonal)
from experiments.exp2_diagonal import run_exp2

best_model = exp1_results.get("best_model_name", "LightGBM")
exp2_results = run_exp2(model_name=best_model)
display(exp2_results["results_df"])

In [ ]:
# Cell 6: Run Experiment 3 — Cross-Generator Transfer Matrix
from experiments.exp3_transfer import run_exp3

exp3_results = run_exp3(model_name=best_model)
print("\n--- 5x5 Cross-Generator Transfer Matrix (F1 Score) ---")
display(exp3_results["df_f1"])

In [ ]:
# Cell 7: Summary & Export Verification
from pathlib import Path

output_dir = Path("outputs/results")
print("Generated Experiment Artifacts:")
for f in sorted(output_dir.glob("**/*")):
    if f.is_file():
        print(f" - {f.relative_to(output_dir)}")